# Locked model feature importance

This notebook interprets the selected **control-aware global linear SRM**. The winning family and feature recipe are fixed before this notebook runs. Out-of-fold performance remains the evidence for generalisation; the full-data fit below is used only for interpretation and future scoring.

## 1. Locked configuration and data

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find repository root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.qc import standardize_train_test
from src.data.trackfa_pairs import trackfa_pairs_to_long
from src.models.srm_global import SRMGlobalLinear
from src.features.panels import a_priori_70_feature_names
from src.reporting.experiment_artifacts import read_experiment_contract, read_table_artifact

RUN_ID = "trackfa_70_feature_comparison_v1"
RUN_DIR = REPO_ROOT / "results" / "experiments" / RUN_ID
SELECTION_DIR = RUN_DIR / "selections"
MODEL_DIR = RUN_DIR / "models" / "srm_global_linear"
OUTPUT_DIR = RUN_DIR / "interpretation" / "srm_global_control_aware"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest, folds = read_experiment_contract(RUN_DIR)
panel = a_priori_70_feature_names()
fold_recipe = read_table_artifact(
    SELECTION_DIR / "control_aware_features_by_fold.csv", schema="feature_recipe", manifest=manifest, panel=panel
)
full_recipe = read_table_artifact(
    SELECTION_DIR / "control_aware_full_data_features.csv", schema="feature_recipe", manifest=manifest, panel=panel
)
fold_coefficients = read_table_artifact(
    MODEL_DIR / "coefficients.csv", schema="coefficients", manifest=manifest
)
fold_parameters = pd.read_csv(MODEL_DIR / "fold_parameters.csv")
performance = read_table_artifact(MODEL_DIR / "performance.csv", schema="performance", manifest=manifest)

selected = full_recipe.loc[full_recipe["selected"].astype(bool)].sort_values("rank", kind="mergesort")
features = selected["feature"].astype(str).tolist()
parameter_counts = (
    fold_parameters.loc[fold_parameters["selection_strategy"].eq("control_aware"), ["ridge", "covariance_shrinkage", "z_clip"]]
    .fillna({"z_clip": "none"}).value_counts().reset_index(name="folds_selected")
    .sort_values(["folds_selected", "covariance_shrinkage"], ascending=[False, False], kind="mergesort")
)
locked = parameter_counts.iloc[0]
RIDGE = float(locked["ridge"])
COVARIANCE_SHRINKAGE = float(locked["covariance_shrinkage"])
Z_CLIP = None if locked["z_clip"] == "none" else float(locked["z_clip"])

display(pd.DataFrame([{
    "Model": "Global linear SRM", "Feature selection": "Control-aware", "Features": len(features),
    "Ridge": RIDGE, "Covariance shrinkage": COVARIANCE_SHRINKAGE,
    "Z clipping": "None" if Z_CLIP is None else Z_CLIP,
    "Interpretation scope": "Full-data refit; not an OOF performance estimate",
}]))
display(performance.loc[
    performance["selection_strategy"].eq("control_aware") & performance["interval"].eq("pooled annual"),
    ["cohort", "n_participants", "n_pairs", "d_z", "ci_low", "ci_high", "p_delta_gt_0"],
].round(3))

,Model,Feature selection,Features,Ridge,Covariance shrinkage,Z clipping,Interpretation scope
0,Global linear SRM,Control-aware,16,0.0,0.45,None,Full-data refit; not an OOF performance estimate


,cohort,n_participants,n_pairs,d_z,ci_low,ci_high,p_delta_gt_0
6,FRDA,117,207,0.749,0.614,0.900,0.768
9,Control,67,126,-0.011,-0.168,0.168,0.484


## 2. Full-data interpretation fit

In [2]:
pairs = pd.read_csv(manifest["data_path"])
frda_long = trackfa_pairs_to_long(pairs)
fit_rows = frda_long[["pair_id", "subject", "visit", *features]].dropna().copy()
X = fit_rows[features].to_numpy(dtype=float)
X_scaled, _, center, scale = standardize_train_test(X, X)
if Z_CLIP is not None:
    X_scaled = np.clip(X_scaled, -Z_CLIP, Z_CLIP)
model = SRMGlobalLinear(ridge=RIDGE, covariance_shrinkage=COVARIANCE_SHRINKAGE).fit(
    X_scaled, fit_rows["pair_id"].to_numpy(), fit_rows["visit"].to_numpy()
)

scoring_parameters = pd.DataFrame({
    "feature": features,
    "training_mean": center,
    "training_sd": scale,
    "standardised_coefficient": model.coef_,
})
scoring_parameters["z_clip"] = Z_CLIP
scoring_parameters.to_csv(OUTPUT_DIR / "new_patient_scoring_parameters.csv", index=False)
print("The deployment fit uses", fit_rows["subject"].nunique(), "FRDA participants and", fit_rows["pair_id"].nunique(), "annual pairs.")

The deployment fit uses 117 FRDA participants and 207 annual pairs.


## 3. Conditional coefficients versus marginal longitudinal change

In [3]:
frequency = (
    fold_recipe.loc[fold_recipe["selected"].astype(bool)].groupby("feature")["outer_fold"]
    .nunique().div(fold_recipe["outer_fold"].nunique()).rename("selection_frequency")
)
fold_summary = (
    fold_coefficients.loc[fold_coefficients["selection_strategy"].eq("control_aware")]
    .groupby("feature")["coefficient"]
    .agg(fold_coefficient_median="median", folds_with_coefficient="size")
)
sign_stability = (
    fold_coefficients.loc[fold_coefficients["selection_strategy"].eq("control_aware")]
    .assign(sign=lambda frame: np.sign(frame["coefficient"]))
    .groupby("feature")["sign"]
    .apply(lambda values: max((values > 0).mean(), (values < 0).mean()))
    .rename("coefficient_sign_stability")
)

importance = selected[[
    "rank", "feature", "selection_score",
    "frda_pooled_d_z", "frda_v1_v2_d_z", "frda_v2_v3_d_z",
    "control_pooled_d_z", "control_v1_v2_d_z", "control_v2_v3_d_z",
]].merge(scoring_parameters, on="feature", how="left")
importance = importance.join(frequency, on="feature").join(fold_summary, on="feature").join(sign_stability, on="feature")
importance["absolute_coefficient"] = importance["standardised_coefficient"].abs()
importance["coefficient_vs_frda_marginal_sign"] = np.where(
    np.sign(importance["standardised_coefficient"]) == np.sign(importance["frda_pooled_d_z"]),
    "same", "different",
)
importance.to_csv(OUTPUT_DIR / "coefficient_marginal_effect_comparison.csv", index=False)

display_table = importance.rename(columns={
    "rank": "Selection rank", "feature": "Feature", "standardised_coefficient": "Conditional coefficient",
    "frda_pooled_d_z": "FRDA pooled d", "control_pooled_d_z": "Control pooled d",
    "frda_v1_v2_d_z": "FRDA V1-V2 d", "frda_v2_v3_d_z": "FRDA V2-V3 d",
    "control_v1_v2_d_z": "Control V1-V2 d", "control_v2_v3_d_z": "Control V2-V3 d",
    "selection_frequency": "Selection frequency", "coefficient_sign_stability": "Coefficient sign stability",
    "coefficient_vs_frda_marginal_sign": "Coefficient/marginal sign",
})
display(display_table[[
    "Selection rank", "Feature", "Conditional coefficient", "FRDA pooled d", "Control pooled d",
    "FRDA V1-V2 d", "FRDA V2-V3 d", "Control V1-V2 d", "Control V2-V3 d",
    "Selection frequency", "Coefficient sign stability", "Coefficient/marginal sign",
]].round(3))

,Selection rank,Feature,Conditional coefficient,FRDA pooled d,Control pooled d,FRDA V1-V2 d,FRDA V2-V3 d,Control V1-V2 d,Control V2-V3 d,Selection frequency,Coefficient sign stability,Coefficient/marginal sign
0,1,Cerebellum_Cortex_CerebNet,-2.176,-0.620,-0.019,-0.878,-0.403,-0.169,0.159,1.0,1.0,same
1,2,Pons,-1.507,-0.552,0.509,-0.777,-0.359,0.411,0.619,1.0,1.0,same
2,3,Cerebellum_WM_CerebNet,-1.471,-0.425,0.228,-0.510,-0.342,0.319,0.142,1.0,1.0,same
3,4,Midbrain,-1.127,-0.352,0.327,-0.388,-0.324,0.303,0.349,1.0,1.0,same
4,5,Thalamus,-0.910,-0.367,0.091,-0.469,-0.248,0.029,0.155,1.0,1.0,same
5,6,FA_PTR,-0.570,-0.253,0.076,-0.207,-0.304,0.256,-0.125,1.0,1.0,same
6,7,Putamen,-0.137,-0.339,-0.155,-0.422,-0.240,-0.117,-0.201,0.8,0.5,same
7,8,RD_SCP,0.524,0.260,-0.085,0.162,0.354,-0.273,0.082,0.8,1.0,same
8,9,Lateral_Ventricle,2.497,0.385,0.249,0.536,0.286,0.283,0.225,1.0,1.0,same
9,10,TotalBrainGMVol_nocereb,-1.069,-0.413,-0.490,-0.402,-0.423,-0.319,-0.686,1.0,1.0,same


## Interpretation of sign differences

The marginal Cohen's (d_z) describes how one MRI feature changes by itself. The model coefficient is a **conditional weight** estimated jointly with correlated MRI features. Consequently, coefficient magnitude or sign can change after shared variance is accounted for.

For example, a feature such as Medulla may have one marginal direction but a different conditional coefficient direction. This is not automatically an error: the coefficient may adjust overlapping information from Pons, Midbrain, or cerebellar measures. Such differences are reported explicitly and should be interpreted as multivariable dependence rather than as a reversal of the feature's biological change.

## 4. New-patient scoring rule

In [4]:
print("For each MRI feature x_j, compute z_j = (x_j - training_mean_j) / training_sd_j.")
print("Apply the frozen clipping rule when present, then calculate score = sum_j(coefficient_j * z_j).")
print("Feature order, means, SDs, coefficients, and clipping are stored in:")
print(OUTPUT_DIR / "new_patient_scoring_parameters.csv")

For each MRI feature x_j, compute z_j = (x_j - training_mean_j) / training_sd_j.
Apply the frozen clipping rule when present, then calculate score = sum_j(coefficient_j * z_j).
Feature order, means, SDs, coefficients, and clipping are stored in:
/Users/robertwang/Documents/New_project/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/interpretation/srm_global_control_aware/new_patient_scoring_parameters.csv
